# SHM Sensor Preprocessing Pipeline (RFL + RFM)

This notebook implements the 8-point preprocessing plan for the structural health
monitoring (SHM) device data, before any anomaly-detection model is touched.

**Data format assumed** (InfluxDB-style export, one row per packet):

```
device, site, server_time, "sensor_id,device_time,<9 numeric fields>,[load cell pairs...],packet_counter"
```

- `RFL` payload numeric fields -> placeholder names `rfl_ch1..rfl_ch9` (tilt/load-related; **rename
  these to your real channel names from the device datasheet** — the parser below is field-name
  agnostic, only the *count* of fields matters), followed by one or more load-cell pairs of the
  form `raw_adc:calib_raw`, `weight_kg`, then a trailing packet counter.
- `RFM` payload numeric fields -> `acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z, mag_x, mag_y, mag_z`
  (this one matches what you described, so it's used directly).

> The RFM row you pasted has an onboard-clock reset (`01/01/2000 00:02:11`) — this is exactly why
> Step 1 below treats `server_time` as the only trustworthy clock and flags `device_time` instead
> of using it for ordering.

> **Field layout is inconsistent across real packets** — `sensor_id` is sometimes absent, and
> `device_date`/`device_time` are sometimes one space-joined token and sometimes two separate
> comma-joined tokens. The parser below detects each piece by its shape (letter-prefixed ->
> sensor_id, `dd/mm/yyyy` -> date, `HH:MM:SS` -> time) rather than by fixed position, so a missing
> field shifts nothing else. `server_time` (from the authoritative InfluxDB timestamp) is also
> split into separate `server_date` / `server_time_of_day` columns.

Sections:
1. Parse raw export -> structured RFL / RFM DataFrames
2. Step 1 — Time alignment (server time as ground truth, fixed-rate resampling, cross-stream sync)
3. Step 2 — Calibration (accel/gyro bias+scale, gyro drift, load-cell tare + temperature)
4. Step 3 — Canonical frame rotation (gravity-vector alignment, static/dynamic split)
5. Step 4 — Denoising (structure-appropriate low-pass, Hampel glitch removal, raw kept alongside filtered)
6. Step 5 — Commissioning baseline (per-node reference stats/PSD)
7. Step 6 — Confound removal (regression residual + cointegration-based residual)
8. Step 7 — Sensor-fault gating (flatline/saturation/packet-loss/RTC-reset/cross-axis coherence)
9. Step 8 — Feature extraction (time + frequency domain, RFL/RFM kept as separate modalities)
10. End-to-end pipeline wrapper


In [1]:
import re
import numpy as np
import pandas as pd
from scipy import signal, stats
from scipy.optimize import least_squares
import warnings

pd.set_option("display.width", 160)


## 0. Load the raw export and parse the packed `value` column

Keep your existing Colab loading cell, then run the parser below on top of it.


In [2]:
# --- Colab loading (kept from your original notebook) ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass  # not running in Colab

RAW_CSV_PATH = '/content/influx_data 2.csv'   # adjust as needed

try:
    df_raw = pd.read_csv(RAW_CSV_PATH, header=None,
                          names=['device', 'site', 'time', 'value'],
                          quotechar='"', engine='python')
except FileNotFoundError:
    warnings.warn(f"{RAW_CSV_PATH} not found — using a small synthetic sample so the rest of "
                   f"the notebook still runs end-to-end. Point RAW_CSV_PATH at your real file.")
    df_raw = pd.DataFrame({
        'device': ['RFL', 'RFM'],
        'site': ['CIDCO', 'CIDCO'],
        'time': ['2026-09-11 10:22:48.491130+00:00', '2026-09-05 06:07:25.364481+00:00'],
        'value': [
            'RFL_01,11/09/2026 15:47:16,-0.56,-0.09,9.41,14.38,-41.75,-22.69,0.06,0.00,0.06,'
            '500:-31197,0.00kg,7.5:-30151,1.85kg,1243',
            'RFM_0001,01/01/2000 00:02:11,0.13,-0.20,-10.02,-0.50,-6.25,3.25,-3.31,-6.19,-2.06,',
        ],
    })

df_raw.head()


/tmp/ipykernel_189/1918585867.py:15: UserWarning: /content/influx_data 2.csv not found — using a small synthetic sample so the rest of the notebook still runs end-to-end. Point RAW_CSV_PATH at your real file.
  warnings.warn(f"{RAW_CSV_PATH} not found — using a small synthetic sample so the rest of "


,device,site,time,value
0,RFL,CIDCO,2026-09-11 10:22:48.491130+00:00,"RFL_01,11/09/2026 15:47:16,-0.56,-0.09,9.41,14..."
1,RFM,CIDCO,2026-09-05 06:07:25.364481+00:00,"RFM_0001,01/01/2000 00:02:11,0.13,-0.20,-10.02..."


In [3]:
RFM_FIELDS = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z', 'mag_x', 'mag_y', 'mag_z']
# TODO: rename to your real RFL channel layout once confirmed against the device datasheet.
RFL_FIELDS = ['rfl_ch1', 'rfl_ch2', 'rfl_ch3', 'rfl_ch4', 'rfl_ch5', 'rfl_ch6', 'rfl_ch7', 'rfl_ch8', 'rfl_ch9']

# Real packets turn out to be inconsistent in two ways:
#  1. `sensor_id` is sometimes missing entirely (payload starts straight with the date/time).
#  2. `device_time` is sometimes ONE token ("dd/mm/yyyy HH:MM:SS", space-separated) and sometimes
#     TWO tokens ("dd/mm/yyyy","HH:MM:SS", comma-separated) -- a naive fixed-position split
#     silently shifts every field after it, which is exactly what put a date string into the
#     `sensor_id` column previously.
_SENSOR_ID_RE = re.compile(r'^[A-Za-z]')
_DATE_RE = re.compile(r'^\d{1,2}/\d{1,2}/\d{4}')
_TIME_ONLY_RE = re.compile(r'^\d{1,2}:\d{2}:\d{2}$')

def _safe_float(tok):
    try:
        return float(tok)
    except (TypeError, ValueError):
        return np.nan

def parse_payload(raw_value, device_type):
    """Parse one packed `value` string into a flat dict, tolerant of the field-shift issues
    above. Layout (all optional/variable-width pieces detected by content, not position):
    [sensor_id], device_date[+ device_time], <9 numeric fields>, [load-cell pairs for RFL], [packet_counter]
    """
    if not isinstance(raw_value, str):
        return {'parse_error': True}
    tokens = raw_value.strip().strip('"').split(',')
    if not tokens:
        return {'parse_error': True}

    idx = 0
    sensor_id = None
    if _SENSOR_ID_RE.match(tokens[0]):
        sensor_id = tokens[0]
        idx = 1

    device_time_raw = None
    if idx < len(tokens) and _DATE_RE.match(tokens[idx]):
        if ' ' in tokens[idx]:
            device_time_raw = tokens[idx]
            idx += 1
        elif idx + 1 < len(tokens) and _TIME_ONLY_RE.match(tokens[idx + 1].strip()):
            device_time_raw = f"{tokens[idx]} {tokens[idx + 1]}"
            idx += 2
        else:
            device_time_raw = tokens[idx]  # date with no matching time token
            idx += 1

    numeric_tokens = tokens[idx:idx + 9]
    numeric = [_safe_float(t) for t in numeric_tokens]
    field_names = RFM_FIELDS if device_type == 'RFM' else RFL_FIELDS

    out = {
        'sensor_id': sensor_id,
        'device_time_raw': device_time_raw,
        'parse_error': (len(numeric_tokens) < 9),
        'nan_field_count': int(np.isnan(numeric).sum()),
    }
    out.update(dict(zip(field_names, numeric)))
    idx += 9

    rest = [t for t in tokens[idx:] if t != '']
    if device_type == 'RFL':
        load_cells = []
        i = 0
        while i < len(rest) - 1:
            a, b = rest[i], rest[i + 1]
            if ':' in a and 'kg' in b:
                try:
                    raw_adc, calib_raw = a.split(':')
                    load_cells.append({
                        'raw_adc': float(raw_adc),
                        'calib_raw': float(calib_raw),
                        'weight_kg_reported': float(b.replace('kg', '')),
                    })
                    i += 2
                    continue
                except ValueError:
                    pass
            i += 1
        out['load_cells'] = load_cells
        counter_tok = rest[-1].strip() if rest else ''
        out['packet_counter'] = int(counter_tok) if counter_tok.lstrip('-').isdigit() else None
        # anything left that isn't a load-cell pair or the trailing counter is unexpected/noise
        consumed = 2 * len(load_cells) + (1 if out['packet_counter'] is not None else 0)
        if consumed < len(rest):
            out['parse_error'] = True
            out['unparsed_tail'] = rest
    return out

def build_device_frame(df_raw, device_type):
    sub = df_raw[df_raw['device'] == device_type].copy()
    parsed = sub['value'].apply(lambda v: parse_payload(v, device_type))
    parsed_df = pd.DataFrame(list(parsed))
    out = pd.concat([sub[['device', 'site', 'time']].reset_index(drop=True),
                      parsed_df.reset_index(drop=True)], axis=1)
    return out

rfl = build_device_frame(df_raw, 'RFL')
rfm = build_device_frame(df_raw, 'RFM')
print('RFL rows:', len(rfl), '| RFM rows:', len(rfm))
rfl.head(), rfm.head()


RFL rows: 1 | RFM rows: 1


(  device   site                              time sensor_id      device_time_raw  parse_error  nan_field_count  rfl_ch1  rfl_ch2  rfl_ch3  rfl_ch4  rfl_ch5  \
 0    RFL  CIDCO  2026-09-11 10:22:48.491130+00:00    RFL_01  11/09/2026 15:47:16        False                0    -0.56    -0.09     9.41    14.38   -41.75   
 
    rfl_ch6  rfl_ch7  rfl_ch8  rfl_ch9                                         load_cells  packet_counter  
 0   -22.69     0.06      0.0     0.06  [{'raw_adc': 500.0, 'calib_raw': -31197.0, 'we...            1243  ,
   device   site                              time sensor_id      device_time_raw  parse_error  nan_field_count  acc_x  acc_y  acc_z  gyro_x  gyro_y  gyro_z  \
 0    RFM  CIDCO  2026-09-05 06:07:25.364481+00:00  RFM_0001  01/01/2000 00:02:11        False                0   0.13   -0.2 -10.02    -0.5   -6.25    3.25   
 
    mag_x  mag_y  mag_z  
 0  -3.31  -6.19  -2.06  )

### Sanity check: the tricky/malformed rows

These three exercise the exact issues seen in practice: a missing `sensor_id` (payload starts
straight with the date), `date` and `time` sent as two separate comma tokens instead of one, and a
malformed/zeroed test packet with a stray empty field. All should parse without the date leaking
into `sensor_id` and without shifting the numeric fields.


In [4]:
_test_rows = [
    ('RFM', '01/09/2026 11:59:58,4.69,-7.84,-3.60,-26.69,42.25,-14.06,-0.06,0.12,0.00'),
    ('RFL', '26/08/2026 14:46:45,00,00,00,00,,00.0,00.0,00.0,00.0,00.0,00.0,00.0'),
    ('RFM', 'RFM_0002,12/09/2026,03:36:40,1.07,-9.44,-0.62,-32.88,5.56,-3.25,0.00,0.06,-0.06,'),
]
for dtype, raw in _test_rows:
    parsed = parse_payload(raw, dtype)
    print(dtype, '->', {k: v for k, v in parsed.items() if k not in ('load_cells',)})
    print()


RFM -> {'sensor_id': None, 'device_time_raw': '01/09/2026 11:59:58', 'parse_error': False, 'nan_field_count': 0, 'acc_x': 4.69, 'acc_y': -7.84, 'acc_z': -3.6, 'gyro_x': -26.69, 'gyro_y': 42.25, 'gyro_z': -14.06, 'mag_x': -0.06, 'mag_y': 0.12, 'mag_z': 0.0}

RFL -> {'sensor_id': None, 'device_time_raw': '26/08/2026 14:46:45', 'parse_error': True, 'nan_field_count': 1, 'rfl_ch1': 0.0, 'rfl_ch2': 0.0, 'rfl_ch3': 0.0, 'rfl_ch4': 0.0, 'rfl_ch5': nan, 'rfl_ch6': 0.0, 'rfl_ch7': 0.0, 'rfl_ch8': 0.0, 'rfl_ch9': 0.0, 'packet_counter': None, 'unparsed_tail': ['00.0', '00.0', '00.0']}

RFM -> {'sensor_id': 'RFM_0002', 'device_time_raw': '12/09/2026 03:36:40', 'parse_error': False, 'nan_field_count': 0, 'acc_x': 1.07, 'acc_y': -9.44, 'acc_z': -0.62, 'gyro_x': -32.88, 'gyro_y': 5.56, 'gyro_z': -3.25, 'mag_x': 0.0, 'mag_y': 0.06, 'mag_z': -0.06}



## Step 1 — Fix time first, before touching values

- `server_time` (from InfluxDB's own `time` column) is ground truth for ordering/windowing.
- `device_time` is parsed too, but only used to *detect* RTC resets (like the RFM row where the
  onboard clock reports year 2000) — never for windowing.
- Each stream is resampled to a strict fixed rate per node. Small gaps in **slow** channels
  (tilt/load) may be interpolated; the vibration channel is never interpolated across a gap for
  frequency-domain features — those windows are marked invalid instead.
- A cross-correlation helper is provided to sub-second-align RFL and RFM around a shared transient
  (e.g. an impact visible in both a load spike and an acceleration spike).


In [5]:
def add_time_columns(df, device_type):
    df = df.copy()
    df['server_time'] = pd.to_datetime(df['time'], utc=True, errors='coerce')
    # server-side date/time, split out as requested -- this is the authoritative clock
    df['server_date'] = df['server_time'].dt.date
    df['server_time_of_day'] = df['server_time'].dt.time

    fmt = '%d/%m/%Y %H:%M:%S'
    df['device_time'] = pd.to_datetime(df['device_time_raw'], format=fmt, errors='coerce')
    df['device_date'] = df['device_time'].dt.date
    df['device_time_of_day'] = df['device_time'].dt.time

    # RTC reset / clearly-wrong device clock -> flag, never trust for ordering
    df['device_clock_bad'] = (
        df['device_time'].isna() | (df['device_time'] < pd.Timestamp('2015-01-01'))
    )
    df = df.sort_values('server_time').reset_index(drop=True)
    return df

rfl = add_time_columns(rfl, 'RFL')
rfm = add_time_columns(rfm, 'RFM')
print('RFM rows with bad onboard clock:', rfm['device_clock_bad'].sum(), '/', len(rfm))


RFM rows with bad onboard clock: 1 / 1


In [6]:
def resample_fixed_rate(df, node_col, time_col, value_cols, freq, slow_channels=None,
                         max_interp_gap='2s'):
    """Resample each node's stream to a strict fixed rate.

    - `slow_channels`: columns allowed a short linear interpolation across small gaps
      (e.g. tilt, load). Anything NOT listed here (vibration/accel/gyro) is left as NaN across a
      gap and the corresponding window must later be marked invalid rather than filled.
    """
    slow_channels = slow_channels or []
    out = []
    for node, g in df.groupby(node_col):
        g = g.set_index(time_col).sort_index()
        g = g[~g.index.duplicated(keep='first')]
        idx = pd.date_range(g.index.min(), g.index.max(), freq=freq)
        r = g[value_cols].reindex(idx, method='nearest', tolerance=pd.Timedelta(freq) / 2)
        gap_mask = r[value_cols].isna().any(axis=1)
        for c in value_cols:
            if c in slow_channels:
                r[c] = r[c].interpolate(limit=int(pd.Timedelta(max_interp_gap) / pd.Timedelta(freq)))
        r['window_gap_flag'] = gap_mask  # True = fill (if any) is synthetic / do not trust for freq-domain use
        r[node_col] = node
        r.index.name = time_col
        out.append(r.reset_index())
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

def align_streams_by_transient(series_a, series_b, fs):
    """Cross-correlate two equal-rate series (e.g. RFL load channel vs RFM accel magnitude)
    around a shared transient and return the lag (seconds) to shift series_b onto series_a."""
    a = np.nan_to_num(series_a.values - np.nanmean(series_a.values))
    b = np.nan_to_num(series_b.values - np.nanmean(series_b.values))
    corr = signal.correlate(a, b, mode='full')
    lag_samples = corr.argmax() - (len(b) - 1)
    return lag_samples / fs


## Step 2 — Calibrate before you filter

- **Accel/gyro bias & scale**: six-position static calibration (or at minimum one long stationary
  capture) solved via least squares against known gravity magnitude.
- **Gyro bias drift**: re-estimated from stationary windows periodically, not just once, since it
  drifts with temperature.
- **Magnetometer** hard/soft-iron correction is stubbed but left off by default (skip unless you
  actually use heading/orientation from `mag_*`).
- **Load cell** tare + temperature compensation, applied before anything else touches the load
  value.


In [7]:
def six_position_calibration(static_captures, g=9.80665):
    """static_captures: dict[str, np.ndarray[N,3]] of stationary accel captures, ideally one per
    axis pointing up/down (6 orientations). Solves per-axis bias `b` and scale `s` such that
    ||(raw - b) * s|| ~= g in every stationary capture. Falls back gracefully with fewer captures
    (accuracy improves with more orientations / longer captures).
    """
    all_samples = np.vstack(list(static_captures.values()))

    def residuals(params):
        bx, by, bz, sx, sy, sz = params
        corrected = (all_samples - [bx, by, bz]) * [sx, sy, sz]
        norms = np.linalg.norm(corrected, axis=1)
        return norms - g

    x0 = [0, 0, 0, 1, 1, 1]
    result = least_squares(residuals, x0)
    bx, by, bz, sx, sy, sz = result.x
    return {'bias': np.array([bx, by, bz]), 'scale': np.array([sx, sy, sz]), 'rmse': np.sqrt(np.mean(result.fun ** 2))}

def estimate_gyro_bias(stationary_gyro_window):
    """stationary_gyro_window: DataFrame/array [N,3] captured while truly still."""
    return np.nanmean(stationary_gyro_window, axis=0)

def apply_accel_calibration(df, cols, bias, scale):
    df = df.copy()
    for i, c in enumerate(cols):
        df[c + '_cal'] = (df[c] - bias[i]) * scale[i]
    return df

def apply_gyro_debias(df, cols, bias):
    df = df.copy()
    for i, c in enumerate(cols):
        df[c + '_cal'] = df[c] - bias[i]
    return df

def load_cell_calibrate(raw_adc, tare_raw, scale_factor, temp=None, ref_temp=None, temp_coeff=None):
    """weight_kg = (raw - tare) * scale_factor, minus a linear temperature correction if a
    co-located temperature reading and manufacturer temp-coefficient are available.
    Re-derive `tare_raw` during known no-load periods, not just once at commissioning.
    """
    weight = (raw_adc - tare_raw) * scale_factor
    if temp is not None and ref_temp is not None and temp_coeff is not None:
        weight = weight - temp_coeff * (temp - ref_temp)
    return weight

# Example usage (fill in with your real commissioning captures):
# calib = six_position_calibration({'up': up_samples, 'down': down_samples, ...})
# rfm = apply_accel_calibration(rfm, ['acc_x', 'acc_y', 'acc_z'], calib['bias'], calib['scale'])


## Step 3 — Rotate into a canonical frame (this is the real fix for "different mounting locations")

1. From a stationary commissioning window, compute the rotation from the device's arbitrary
   mounting orientation into a canonical structural frame using the measured gravity vector.
2. Apply that per-node rotation matrix to every subsequent sample, before feature extraction.
3. Separate the static/gravity component (long-term mean -> absolute tilt/inclination) from the
   dynamic vibration component (high-pass -> oscillation signal) — these are two different
   physical phenomena and should not be conflated into one feature set.


In [8]:
def rotation_from_gravity(stationary_accel_mean, g_ref=np.array([0, 0, 1.0])):
    """Rotation matrix R such that R @ stationary_accel_mean (normalized) ~= g_ref.
    Uses Rodrigues' rotation formula between the two unit vectors.
    """
    v = stationary_accel_mean / np.linalg.norm(stationary_accel_mean)
    g = g_ref / np.linalg.norm(g_ref)
    axis = np.cross(v, g)
    s = np.linalg.norm(axis)
    c = np.dot(v, g)
    if s < 1e-8:
        return np.eye(3) if c > 0 else -np.eye(3)
    axis = axis / s
    K = np.array([[0, -axis[2], axis[1]],
                  [axis[2], 0, -axis[0]],
                  [-axis[1], axis[0], 0]])
    R = np.eye(3) + K * s + K @ K * (1 - c)
    return R

def apply_rotation(df, cols, R, out_prefix='rot_'):
    df = df.copy()
    mat = df[cols].to_numpy()
    rotated = mat @ R.T
    for i, c in enumerate(cols):
        df[out_prefix + c] = rotated[:, i]
    return df

def static_dynamic_split(series, fs, cutoff_hz=0.3, order=4):
    """Static (gravity/tilt) = low-frequency component; dynamic (vibration) = high-pass residual."""
    b, a = signal.butter(order, cutoff_hz / (fs / 2), btype='low')
    static = signal.filtfilt(b, a, series.ffill().bfill())
    dynamic = series.values - static
    return pd.Series(static, index=series.index), pd.Series(dynamic, index=series.index)

# Per-node rotation store: {node_id: R}
NODE_ROTATIONS = {}
# Example: NODE_ROTATIONS['RFM_0001'] = rotation_from_gravity(stationary_mean_accel_for_that_node)


## Step 4 — Denoise with a frequency budget tied to the structure, not a generic default

- Low-pass cutoff set from the structure's actual natural frequencies of interest (bridges/
  flyovers: typically well under 20–30 Hz for the modes that matter) — set `STRUCTURE_CUTOFF_HZ`
  per site/structure type.
- Hampel filter for single-sample glitch removal, window sized so it doesn't smear out a genuine
  short transient (e.g. a rockfall impact).
- Raw and filtered columns are both kept — never destructively overwrite raw.


In [9]:
STRUCTURE_CUTOFF_HZ = 25.0  # TODO: set from the actual modal frequencies of interest for this structure

def lowpass_filter(series, fs, cutoff_hz=STRUCTURE_CUTOFF_HZ, order=4):
    nyq = fs / 2
    cutoff = min(cutoff_hz, nyq * 0.99)
    b, a = signal.butter(order, cutoff / nyq, btype='low')
    filled = series.ffill().bfill()
    return pd.Series(signal.filtfilt(b, a, filled), index=series.index)

def hampel_filter(series, window_size=7, n_sigmas=3):
    """Median-based glitch (single-sample outlier) removal. `window_size` should be small enough
    that it cannot eat a genuine short transient — tune against your shortest real event duration.
    """
    k = 1.4826  # scale factor for Gaussian consistency
    rolling_median = series.rolling(window_size, center=True).median()
    diff = (series - rolling_median).abs()
    mad = diff.rolling(window_size, center=True).median()
    threshold = n_sigmas * k * mad
    outlier_mask = diff > threshold
    cleaned = series.copy()
    cleaned[outlier_mask] = rolling_median[outlier_mask]
    return cleaned, outlier_mask

def denoise_columns(df, cols, fs, hampel_window=7, hampel_sigmas=3):
    df = df.copy()
    for c in cols:
        clean, glitch_mask = hampel_filter(df[c], hampel_window, hampel_sigmas)
        df[c + '_filt'] = lowpass_filter(clean, fs)
        df[c + '_glitch'] = glitch_mask
        # df[c] itself remains the untouched raw column
    return df


## Step 5 — Build a real commissioning baseline, not a rolling window from day one

- An explicit healthy calibration period per node (days, not minutes) is required before scoring
  goes live.
- Per node: mean/std of each feature, reference PSD shape, reference dominant frequency are stored.
- Everything afterward is scored as a deviation from *that specific node's* reference, never a
  fleet-wide average.


In [10]:
def build_node_baseline(node_df, feature_cols, fs, welch_nperseg=256):
    """node_df: preprocessed data for ONE node, restricted to its healthy commissioning window."""
    baseline = {'mean': {}, 'std': {}, 'psd_ref': {}, 'dominant_freq': {}}
    for c in feature_cols:
        vals = node_df[c].dropna()
        baseline['mean'][c] = float(vals.mean())
        baseline['std'][c] = float(vals.std())
        if len(vals) >= welch_nperseg:
            f, pxx = signal.welch(vals, fs=fs, nperseg=welch_nperseg)
            baseline['psd_ref'][c] = {'f': f.tolist(), 'pxx': pxx.tolist()}
            baseline['dominant_freq'][c] = float(f[np.argmax(pxx)])
    return baseline

# BASELINES: {node_id: {feature_col: {...}}}  -- build once per node over its commissioning window,
# then persist (e.g. json.dump) so it survives restarts.
BASELINES = {}


## Step 6 — Strip out environmental/operational confounds before scoring

Temperature, traffic load and time-of-day all shift features in ways that look like damage but
aren't. Two options, from simplest to most rigorous:

1. **Regression residual**: regress the damage-sensitive feature against a confound proxy
   (temperature if available, else time-of-day/season harmonic terms) fit on the healthy baseline
   period; work with the residual, not the raw feature.
2. **Cointegration** (Engle–Granger two-step): combine the non-stationary vibration feature with
   the confounding variable so the linear combination is stationary, and use that residual as the
   damage index — this is what's actually used in the SHM literature for separating "damaged" from
   "hot".


In [11]:
def harmonic_time_features(timestamps, periods_hours=(24, 24 * 365.25)):
    """Fourier terms for time-of-day / seasonal periodicity, used as a confound proxy when no
    temperature sensor is available."""
    t = (timestamps - timestamps.min()).total_seconds() / 3600.0  # hours since start
    feats = {}
    for p in periods_hours:
        feats[f'sin_{p:g}h'] = np.sin(2 * np.pi * t / p)
        feats[f'cos_{p:g}h'] = np.cos(2 * np.pi * t / p)
    return pd.DataFrame(feats, index=timestamps.index if hasattr(timestamps, 'index') else None)

def regress_out_confound(feature, confound_df, healthy_mask):
    """OLS fit on the healthy period only; residual computed over the whole series."""
    X_fit = confound_df.loc[healthy_mask].to_numpy()
    y_fit = feature.loc[healthy_mask].to_numpy()
    X_fit_ = np.column_stack([np.ones(len(X_fit)), X_fit])
    coef, *_ = np.linalg.lstsq(X_fit_, y_fit, rcond=None)

    X_all_ = np.column_stack([np.ones(len(confound_df)), confound_df.to_numpy()])
    pred = X_all_ @ coef
    residual = feature.to_numpy() - pred
    return pd.Series(residual, index=feature.index), coef

def cointegration_residual(y, x):
    """Engle-Granger two-step: OLS hedge ratio y ~ x, residual = y - beta*x - alpha.
    Use scipy/numpy directly to avoid a hard statsmodels dependency; swap in
    `statsmodels.tsa.stattools.coint` for a formal stationarity test on the residual if available.
    """
    x_ = np.column_stack([np.ones(len(x)), x])
    coef, *_ = np.linalg.lstsq(x_, y, rcond=None)
    alpha, beta = coef
    residual = y - (alpha + beta * x)
    return residual, {'alpha': alpha, 'beta': beta}

try:
    from statsmodels.tsa.stattools import coint as sm_coint

    def coint_test(y, x):
        """Returns (test_stat, p_value, crit_values). p < 0.05 -> series are cointegrated, i.e. the
        residual is a valid stationary damage index."""
        return sm_coint(y, x)
except ImportError:
    def coint_test(y, x):
        warnings.warn('statsmodels not installed — skipping formal cointegration test; '
                       'the Engle-Granger residual above is still usable as a damage index.')
        return None


## Step 7 — Gate for sensor faults before structural anomaly scoring

Treat "sensor is broken" as a **separate class** from "structure is anomalous" — conflating them
is the single biggest precision killer in low-cost SHM systems.

- Flatlined/stuck values
- Out-of-range / saturated readings
- Abnormal packet-loss rate
- Timestamp/RTC resets (from Step 1)
- Cross-axis/cross-sensor coherence: a real structural event tends to show correlated response
  across multiple axes/sensors; a single miscalibrated or failing sensor typically doesn't.


In [12]:
def detect_flatline(series, window=20, tol=1e-6):
    return series.rolling(window).std() < tol

def detect_saturation(series, sensor_min, sensor_max, margin=0.01):
    span = sensor_max - sensor_min
    return (series <= sensor_min + margin * span) | (series >= sensor_max - margin * span)

def detect_packet_loss(server_time, expected_interval_s, tolerance=2.0):
    dt = server_time.diff().dt.total_seconds()
    return dt > expected_interval_s * tolerance

def cross_axis_coherence(df, cols, window=50, corr_threshold=0.3):
    """Low rolling mean pairwise correlation across the given columns suggests the anomaly is
    isolated to one sensor/axis rather than a real structural response."""
    roll_corr = df[cols].rolling(window).corr()
    mean_abs_corr = (
        roll_corr.abs()
        .groupby(level=0)
        .apply(lambda m: (m.to_numpy().sum() - len(cols)) / (len(cols) ** 2 - len(cols)) if len(cols) > 1 else np.nan)
    )
    return mean_abs_corr < corr_threshold  # True = likely isolated single-sensor issue

def build_fault_flags(df, value_cols, server_time_col, sensor_ranges, expected_interval_s):
    flags = pd.DataFrame(index=df.index)
    for c in value_cols:
        flags[c + '_flatline'] = detect_flatline(df[c])
        if c in sensor_ranges:
            lo, hi = sensor_ranges[c]
            flags[c + '_saturated'] = detect_saturation(df[c], lo, hi)
    flags['packet_loss'] = detect_packet_loss(df[server_time_col], expected_interval_s)
    if 'device_clock_bad' in df.columns:
        flags['rtc_reset'] = df['device_clock_bad']
    flags['isolated_sensor_deviation'] = cross_axis_coherence(df, value_cols)
    flags['any_fault'] = flags.any(axis=1)
    return flags

# Example: sensor_ranges = {'acc_x': (-16, 16), 'acc_y': (-16, 16), 'acc_z': (-16, 16)}  # in g, per datasheet


## Step 8 — Feature extraction (after everything above, not instead of it)

- **Time-domain**: RMS, peak, crest factor, kurtosis (sensitive to impulsive/impact events like a
  rockfall), skewness — computed on windows covering several cycles of the lowest frequency of
  interest, ~50% overlap.
- **Frequency-domain**: Welch's method (lower-variance PSD than a raw FFT); track dominant
  frequency and its amplitude over time — a persistent frequency shift beyond the noise floor is
  the most physically meaningful damage indicator available.
- **RFL (quasi-static: load/tilt)** and **RFM (dynamic: vibration)** are extracted as separate
  modalities with separate baselines, fused at the decision/ensemble level rather than
  concatenated into one feature vector, since they represent different physical regimes.


In [13]:
def time_domain_features(window):
    w = window.dropna()
    if len(w) < 2:
        return {k: np.nan for k in ['rms', 'peak', 'crest_factor', 'kurtosis', 'skewness']}
    rms = np.sqrt(np.mean(w ** 2))
    peak = np.max(np.abs(w))
    return {
        'rms': rms,
        'peak': peak,
        'crest_factor': peak / rms if rms > 0 else np.nan,
        'kurtosis': stats.kurtosis(w),
        'skewness': stats.skew(w),
    }

def freq_domain_features(window, fs, nperseg=256):
    w = window.dropna()
    if len(w) < nperseg:
        return {'dominant_freq': np.nan, 'dominant_amp': np.nan}
    f, pxx = signal.welch(w, fs=fs, nperseg=nperseg)
    idx = np.argmax(pxx)
    return {'dominant_freq': f[idx], 'dominant_amp': pxx[idx]}

def windowed_features(series, fs, window_s, overlap=0.5, freq_domain=True):
    win_len = int(window_s * fs)
    step = max(1, int(win_len * (1 - overlap)))
    rows = []
    for start in range(0, max(1, len(series) - win_len + 1), step):
        w = series.iloc[start:start + win_len]
        feats = time_domain_features(w)
        if freq_domain:
            feats.update(freq_domain_features(w, fs))
        feats['window_start'] = w.index[0] if hasattr(w, 'index') and len(w) else None
        rows.append(feats)
    return pd.DataFrame(rows)

def extract_rfm_features(node_df, fs, window_s=10, overlap=0.5, axis_cols=('rot_acc_x', 'rot_acc_y', 'rot_acc_z')):
    """Dynamic/vibration modality (RFM): windowed time+frequency features per axis."""
    out = {}
    for c in axis_cols:
        if c in node_df.columns:
            out[c] = windowed_features(node_df[c], fs, window_s, overlap, freq_domain=True)
    return out

def extract_rfl_features(node_df, fs, window_s=60, overlap=0.5, quasi_static_cols=('rot_rfl_ch1',)):
    """Quasi-static modality (RFL): longer windows, time-domain + slow trend, no high-freq PSD needed."""
    out = {}
    for c in quasi_static_cols:
        if c in node_df.columns:
            out[c] = windowed_features(node_df[c], fs, window_s, overlap, freq_domain=False)
    return out

def fuse_decision(rfl_anomaly_score, rfm_anomaly_score, mode='max'):
    """Combine per-modality anomaly scores at the decision level (never concatenate raw feature
    vectors across RFL/RFM — one modality's variance would dominate the other's)."""
    if mode == 'max':
        return max(rfl_anomaly_score, rfm_anomaly_score)
    if mode == 'weighted':
        return 0.5 * rfl_anomaly_score + 0.5 * rfm_anomaly_score
    raise ValueError(mode)


## End-to-end pipeline wrapper

Chains steps 1–8 for one node's data. Commissioning-time inputs you still need to supply per node:
- a stationary six-position (or long single-position) capture for `six_position_calibration`
- a stationary window for `rotation_from_gravity`
- a known-healthy multi-day window for `build_node_baseline`
- sensor min/max ranges and expected sampling interval for `build_fault_flags`


In [14]:
def run_preprocessing_pipeline(node_df, device_type, fs, node_id,
                                accel_cols=None, calib=None, rotation=None,
                                sensor_ranges=None, expected_interval_s=None,
                                healthy_mask=None, confound_series=None):
    """Runs Steps 1-8 for a single node's already-time-sorted DataFrame.
    Returns (processed_df, fault_flags, baseline, features).
    """
    df = node_df.copy()

    # Step 2: calibration (if params supplied)
    if accel_cols and calib is not None:
        df = apply_accel_calibration(df, accel_cols, calib['bias'], calib['scale'])
        cal_cols = [c + '_cal' for c in accel_cols]
    else:
        cal_cols = accel_cols or []

    # Step 3: canonical-frame rotation + static/dynamic split
    if rotation is not None and cal_cols:
        df = apply_rotation(df, cal_cols, rotation)
        rot_cols = ['rot_' + c for c in cal_cols]
        for c in rot_cols:
            static, dynamic = static_dynamic_split(df[c], fs)
            df[c + '_static'] = static
            df[c + '_dynamic'] = dynamic
    else:
        rot_cols = cal_cols

    # Step 4: denoise
    if rot_cols:
        df = denoise_columns(df, rot_cols, fs)

    # Step 6: confound removal (example on the first rotated column, if present + healthy_mask given)
    if rot_cols and healthy_mask is not None and confound_series is not None:
        resid, _ = regress_out_confound(df[rot_cols[0]], pd.DataFrame({'confound': confound_series}), healthy_mask)
        df[rot_cols[0] + '_confound_resid'] = resid

    # Step 7: fault gating
    fault_flags = None
    if sensor_ranges and expected_interval_s and rot_cols:
        fault_flags = build_fault_flags(df, rot_cols, 'server_time', sensor_ranges, expected_interval_s)

    # Step 5: baseline (only meaningful restricted to the healthy commissioning window)
    baseline = None
    if healthy_mask is not None and rot_cols:
        baseline = build_node_baseline(df.loc[healthy_mask], rot_cols, fs)

    # Step 8: features
    features = None
    if rot_cols:
        if device_type == 'RFM':
            features = extract_rfm_features(df, fs, axis_cols=tuple(rot_cols))
        else:
            features = extract_rfl_features(df, fs, quasi_static_cols=tuple(rot_cols))

    return df, fault_flags, baseline, features


## Step 9 — Layered anomaly-scoring ensemble (advanced SHM coverage)

A single model can't cover all three failure signatures that matter here — a real vibration
event, a slow fence/device shift, and a device fault are physically different things, and
conflating them into one model is exactly what caps precision in most low-cost SHM systems
(commercial rockfall-fence systems keep load and motion monitoring separate for the same
reason). This section implements the four-layer architecture:

| Layer | Detects | Tool |
|---|---|---|
| 0. Fault gate | Device fluctuation / sensor fault | `fault_flags` from Step 7 |
| 1. Vibration anomaly | Real transient events (impacts, unusual oscillation) | LSTM-autoencoder on RFM windows |
| 2. Slow-shift detector | Fence/device physically moved over time | CUSUM vs. the **fixed** Step 5 commissioning baseline |
| 3. Load-event detector | Something is resting on / has struck the fence | Threshold + rate-of-change on tared RFL load channels |

**Decision rule**: if Layer 0 fires, report a device fault and suppress the structural alert —
a faulty sensor must never be scored as a structural event. Otherwise, alert if Layer 1, 2,
or 3 fires. The layers are fused at the decision level, not concatenated into one feature
vector, for the same reason RFL/RFM are kept as separate modalities in Step 8.

### Layer 2 — CUSUM slow-shift detector

Anchored to Step 5's **fixed** commissioning baseline, never a rolling window — a rolling
baseline would let a slow permanent shift (a leaning post, load-cell creep) get relearned as
"normal" before it ever triggers an alarm, which is the known blind spot of models that adapt
to recent data. `k` (allowance) and `h` (decision threshold), both in std-dev units, must be
tuned against a real healthy period to hit your target false-alarm rate — the values below are
starting points, not final settings.

In [15]:
def cusum_detector(series, baseline_mean, baseline_std, k=0.5, h=5.0):
    """Two-sided CUSUM control chart for a persistent shift away from a FIXED baseline.
    series: 1D array-like (already time-ordered).
    baseline_mean / baseline_std: from Step 5's `build_node_baseline` for this exact channel,
        e.g. `baseline['mean'][col]`, `baseline['std'][col]` -- NOT recomputed from `series`
        itself, or a real drift would just shift the baseline along with it.
    k: allowance, in std-dev units -- how much drift to tolerate before it starts accumulating
        (commonly ~0.5x the size of the shift, in std devs, that you want to be sensitive to).
    h: decision threshold, in std-dev units -- alarm when the cumulative sum exceeds h.
    Returns (alarm: bool array, pos: running positive CUSUM, neg: running negative CUSUM).
    """
    z = (np.asarray(series, dtype=float) - baseline_mean) / baseline_std
    pos = np.zeros(len(z))
    neg = np.zeros(len(z))
    for i in range(1, len(z)):
        pos[i] = max(0.0, pos[i - 1] + z[i] - k)
        neg[i] = min(0.0, neg[i - 1] + z[i] + k)
    alarm = (pos > h) | (neg < -h)
    return alarm, pos, neg


def slow_shift_flags(node_df, cols, baseline, k=0.5, h=5.0):
    """Runs `cusum_detector` per column against that column's Step 5 baseline stats.
    Returns a DataFrame of per-column alarm flags plus `any_slow_shift`.
    """
    out = pd.DataFrame(index=node_df.index)
    for c in cols:
        if c in baseline.get('mean', {}) and c in node_df.columns:
            alarm, _, _ = cusum_detector(
                node_df[c].ffill().bfill(), baseline['mean'][c], baseline['std'][c] or 1e-8, k=k, h=h
            )
            out[c + '_slow_shift'] = alarm
    out['any_slow_shift'] = out.any(axis=1) if len(out.columns) else False
    return out

### Layer 3 — RFL load-event detector

A load-cell reading rising above its tared baseline is a direct, physically unambiguous signal
that something is resting on or has struck the fence — a deep model is the wrong tool for this
specific channel; a threshold/rate-of-change check on the already-tared value (Step 2) is both
more precise and more interpretable.

In [16]:
def load_event_detector(load_kg, tare_kg=0.0, abs_threshold_kg=None,
                         rate_threshold_kg_per_s=None, fs=None):
    """load_kg: already-tared/temperature-compensated load series (Step 2's `load_cell_calibrate`).
    abs_threshold_kg: flag if |load - tare| exceeds this (a sustained load is now present).
    rate_threshold_kg_per_s + fs: flag if the load is changing faster than this (an impact/sudden
        addition, as opposed to slow creep already handled by Layer 2's CUSUM on the load channel).
    Both are optional and can be combined; set thresholds from the load cell's rated
    resolution/noise floor, not an arbitrary round number.
    """
    load_kg = np.asarray(load_kg, dtype=float)
    delta = load_kg - tare_kg
    flags = np.zeros(len(delta), dtype=bool)
    if abs_threshold_kg is not None:
        flags |= np.abs(delta) > abs_threshold_kg
    if rate_threshold_kg_per_s is not None and fs is not None:
        rate = np.diff(delta, prepend=delta[0]) * fs
        flags |= np.abs(rate) > rate_threshold_kg_per_s
    return flags, delta

### Layer 1 — LSTM-autoencoder for RFM vibration anomalies

Trained only on healthy windows from the Step 5 commissioning period; reconstruction error on
new windows becomes the vibration anomaly score. A few things matter here for precision, more
than architecture choice does:

- **Normalize each channel with Step 5's baseline mean/std before windowing.** Accel, gyro, and
  (if used) magnetometer sit on very different scales — an unnormalized model spends its
  capacity fitting whichever channel happens to have the largest raw magnitude.
- **Window length** should cover several cycles of the lowest frequency you care about (same
  rule as Step 8's feature windows): too short and it can't see the vibration pattern, too long
  and a brief impact gets diluted into the average.
- **This layer only answers "is the vibration pattern unusual."** It does not know a faulty
  sensor from a real event (that's Layer 0's job) and will not reliably catch a slow drift if
  you ever retrain it on a rolling window (that's why Layer 2 is a separate, fixed-baseline
  detector, not folded into this model).
- Requires `torch` (`pip install torch`).

In [17]:
import torch
import torch.nn as nn


class LSTMAutoencoder(nn.Module):
    """Minimal sequence-to-sequence LSTM autoencoder: encodes a window to a latent vector,
    decodes back to the full window length. Reconstruction error = vibration anomaly score.
    """
    def __init__(self, n_features, seq_len, hidden_dim=32, latent_dim=8):
        super().__init__()
        self.seq_len = seq_len
        self.encoder = nn.LSTM(n_features, hidden_dim, batch_first=True)
        self.encoder_fc = nn.Linear(hidden_dim, latent_dim)
        self.decoder_fc = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.LSTM(hidden_dim, n_features, batch_first=True)

    def forward(self, x):
        _, (h_n, _) = self.encoder(x)
        z = self.encoder_fc(h_n[-1])
        h_dec = self.decoder_fc(z).unsqueeze(1).repeat(1, self.seq_len, 1)
        out, _ = self.decoder(h_dec)
        return out


def make_windows(arr, seq_len, step):
    """arr: [T, n_features] -> windows: [n_windows, seq_len, n_features], plus each window's
    start index (needed to map anomaly scores back to timestamps)."""
    windows, starts = [], []
    for start in range(0, len(arr) - seq_len + 1, step):
        windows.append(arr[start:start + seq_len])
        starts.append(start)
    shape = (0, seq_len, arr.shape[1])
    return (np.stack(windows) if windows else np.empty(shape)), starts


def normalize_windows_with_baseline(windows, cols, baseline):
    """Uses Step 5's per-node baseline mean/std (not stats recomputed from `windows` itself,
    or the model would normalize away exactly the drift Layer 2 is meant to catch)."""
    mean = np.array([baseline['mean'][c] for c in cols])
    std = np.array([baseline['std'][c] or 1e-8 for c in cols])
    return (windows - mean) / std


def train_lstm_autoencoder(healthy_windows, hidden_dim=32, latent_dim=8, epochs=30,
                            lr=1e-3, batch_size=32, device='cpu', verbose=True):
    """healthy_windows: normalized [n_windows, seq_len, n_features], from the commissioning
    period only."""
    n_features, seq_len = healthy_windows.shape[2], healthy_windows.shape[1]
    model = LSTMAutoencoder(n_features, seq_len, hidden_dim, latent_dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    X = torch.tensor(healthy_windows, dtype=torch.float32).to(device)
    for epoch in range(epochs):
        perm = torch.randperm(len(X))
        total = 0.0
        for i in range(0, len(X), batch_size):
            idx = perm[i:i + batch_size]
            batch = X[idx]
            opt.zero_grad()
            recon = model(batch)
            loss = loss_fn(recon, batch)
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        if verbose and (epoch % max(1, epochs // 5) == 0 or epoch == epochs - 1):
            print(f"epoch {epoch + 1}/{epochs} - recon MSE {total / len(X):.5f}")
    return model


def lstm_reconstruction_scores(model, windows, device='cpu'):
    """windows: normalized, any set (healthy or live). Returns per-window MSE."""
    model.eval()
    with torch.no_grad():
        X = torch.tensor(windows, dtype=torch.float32).to(device)
        recon = model(X)
        return ((recon - X) ** 2).mean(dim=(1, 2)).cpu().numpy()


def lstm_anomaly_threshold(healthy_scores, n_sigmas=4):
    """Set from the healthy period's OWN score distribution, not a fixed absolute number --
    every node's baseline vibration level differs."""
    return healthy_scores.mean() + n_sigmas * healthy_scores.std()

### Final decision — fuse all four layers

In [18]:
def layered_anomaly_decision(fault_flag, vibration_anomaly, slow_shift_flag, load_event_flag):
    """Layer 0 gates everything: a fault suppresses the structural alert entirely, since a
    faulty sensor must never be scored as a structural event. Otherwise, alert if ANY of
    Layers 1-3 fire, and report which one(s) so the alert is attributable, not just a score.
    """
    if fault_flag:
        return {'status': 'device_fault', 'structural_alert': False, 'triggers': []}
    triggers = [name for name, v in [
        ('vibration', vibration_anomaly),
        ('slow_shift', slow_shift_flag),
        ('load_event', load_event_flag),
    ] if v]
    return {'status': 'alert' if triggers else 'normal',
            'structural_alert': bool(triggers), 'triggers': triggers}

### Sanity check: the ensemble on synthetic data

Same convention as the parser sanity check earlier — synthetic but structurally realistic data
that exercises each layer independently: a clean vibration signal with one injected impact
burst (Layer 1), a slow linear drift after a stable period (Layer 2), and a step-change in load
(Layer 3). This is here so the pipeline can be verified end-to-end without needing your real
CSV; swap in `rot_acc_x_filt` / `rot_rfl_ch*_filt` etc. from `run_preprocessing_pipeline`'s
output once you have enough real commissioning data to train the LSTM-AE.

In [19]:
np.random.seed(0)
torch.manual_seed(0)

# --- Layer 1 demo: healthy 3-axis vibration + one injected impact burst ---
fs_demo = 50.0
t_demo = np.arange(0, 200, 1 / fs_demo)
sig = np.stack([
    0.05 * np.sin(2 * np.pi * 2.0 * t_demo) + 0.02 * np.random.randn(len(t_demo)),
    0.03 * np.sin(2 * np.pi * 3.0 * t_demo + 0.5) + 0.02 * np.random.randn(len(t_demo)),
    0.04 * np.sin(2 * np.pi * 2.5 * t_demo + 1.0) + 0.02 * np.random.randn(len(t_demo)),
], axis=1)
burst_start, burst_len = int(150 * fs_demo), int(0.5 * fs_demo)
sig[burst_start:burst_start + burst_len] += np.random.randn(burst_len, 3) * 1.5

demo_cols = ['rot_acc_x_filt', 'rot_acc_y_filt', 'rot_acc_z_filt']
seq_len, step = int(2 * fs_demo), int(0.5 * fs_demo)
windows, starts = make_windows(sig, seq_len, step)
healthy_mask = np.array([s + seq_len <= burst_start for s in starts])

demo_baseline = {'mean': {c: sig[:burst_start, i].mean() for i, c in enumerate(demo_cols)},
                  'std': {c: sig[:burst_start, i].std() for i, c in enumerate(demo_cols)}}
train_windows = normalize_windows_with_baseline(windows[healthy_mask], demo_cols, demo_baseline)
all_windows = normalize_windows_with_baseline(windows, demo_cols, demo_baseline)

model = train_lstm_autoencoder(train_windows, hidden_dim=16, latent_dim=6, epochs=20, verbose=False)
scores = lstm_reconstruction_scores(model, all_windows)
thresh = lstm_anomaly_threshold(scores[healthy_mask])
burst_windows = [i for i, s in enumerate(starts) if s <= burst_start < s + seq_len]
print(f"Layer 1 (vibration) -- threshold: {thresh:.4f} | score at injected impact: "
      f"{scores[burst_windows].max():.4f} | flagged {int((scores > thresh).sum())}/{len(scores)} windows")

# --- Layer 2 demo: stable then slowly drifting tilt ---
tilt_demo = np.concatenate([np.random.randn(1000) * 0.05, np.linspace(0, 1.5, 500) + np.random.randn(500) * 0.05])
alarm, pos, neg = cusum_detector(tilt_demo, baseline_mean=0.0, baseline_std=0.05, k=0.5, h=10.0)
first_alarm = np.argmax(alarm) if alarm.any() else None
print(f"Layer 2 (slow shift) -- drift starts at sample 1000, first alarm at sample {first_alarm}")

# --- Layer 3 demo: step-change in load (something now resting on the fence) ---
load_demo = np.concatenate([np.zeros(50), np.linspace(0, 1.85, 5), np.full(50, 1.85)])
load_flags, load_delta = load_event_detector(load_demo, tare_kg=0.0, abs_threshold_kg=0.3)
print(f"Layer 3 (load event) -- first flagged sample: {np.argmax(load_flags) if load_flags.any() else None}")

# --- fused decision for a few example moments ---
print()
print('during the impact burst  ->', layered_anomaly_decision(False, True, False, False))
print('after the slow drift     ->', layered_anomaly_decision(False, False, True, False))
print('load newly on the fence  ->', layered_anomaly_decision(False, False, False, True))
print('device fault present     ->', layered_anomaly_decision(True, True, True, True))

Layer 1 (vibration) -- threshold: 1.2511 | score at injected impact: 451.6483 | flagged 4/397 windows
Layer 2 (slow shift) -- drift starts at sample 1000, first alarm at sample 1019
Layer 3 (load event) -- first flagged sample: 51

during the impact burst  -> {'status': 'alert', 'structural_alert': True, 'triggers': ['vibration']}
after the slow drift     -> {'status': 'alert', 'structural_alert': True, 'triggers': ['slow_shift']}
load newly on the fence  -> {'status': 'alert', 'structural_alert': True, 'triggers': ['load_event']}
device fault present     -> {'status': 'device_fault', 'structural_alert': False, 'triggers': []}


---
### Summary

Steps 1-8 (parsing through feature extraction) plus Step 9's four-layer ensemble
(fault gate -> LSTM-autoencoder for vibration -> CUSUM for slow shift -> threshold/rate for
load events -> fused decision) are all implemented above and exercised against synthetic data
in each section's sanity check.

**Still yours to fill in against your real deployment** -- none of this can be invented, it has
to come from your actual hardware/site:
- Real commissioning captures (six-position accel calibration, stationary gyro/orientation
  window, multi-day healthy baseline) per node, feeding Steps 2/3/5.
- Sensor min/max ranges and expected sampling interval, for Step 7's fault gate.
- CUSUM's `k`/`h` (Layer 2) and the load thresholds (Layer 3), tuned against real healthy data
  to hit your target false-alarm rate.
- Confirming the RFL channel meanings (`rfl_ch1..9`) against the device datasheet, once you
  have it, so Steps 5/8/9's column names mean something physically specific instead of
  placeholders.